# BPL Dots

A small, self-contained extension for displaying BPL arrays as dots. Run the setup below once, then try the examples. All the code is in this dialogue; no helper files or connection to Scratchpad are needed.

## Setup

Tested with BasedPL 0.1.82. The setup needs BasedPL, NumPy and IPython. If BasedPL is missing, run `%pip install basedpl==0.1.82 numpy` in a Python code cell first. Rerun the setup after a kernel restart. Saved outputs can be viewed without running code.

In [ ]:
"""Self-contained BPL Dots setup. Run once per notebook kernel."""

import html as _df_html
import math as _df_math

import numpy as _df_np
from IPython.display import HTML as _DFHTML
from basedpl import bpl as _df_bpl


def _df_value(source):
    value = _df_bpl(source) if isinstance(source, str) else source
    raw = value.np if hasattr(value, "np") else value
    array = _df_np.asarray(raw)
    if array.dtype.kind not in "biufc":
        try:
            array = array.astype(float)
        except (ValueError, TypeError) as exc:
            raise ValueError("Use a rectangular numeric array for this prototype.") from exc
    if array.size > 4096:
        raise ValueError("Choose an example with at most 4096 values.")
    return array


def _df_number(value):
    value = value.item() if hasattr(value, "item") else value
    if isinstance(value, bool):
        return "$t" if value else "$f"
    if isinstance(value, complex):
        return f"{value.real:.17g}j{value.imag:.17g}".replace("-", "¯")
    if isinstance(value, int):
        return str(value).replace("-", "¯")
    return format(value, ".17g").replace("-", "¯")




import argparse as _dm_argparse
import shlex as _dm_shlex
from IPython import get_ipython as _dm_get_ipython
from IPython.core.error import UsageError as _DMUsageError
from IPython.display import display as _dm_display
import basedpl as _dm_bp
import basedpl.notebooks as _dm_notebooks
from basedpl.syntax import parse as _dm_parse


class _DotMagicParser(_dm_argparse.ArgumentParser):
    def error(self, message):
        raise _DMUsageError(message)


def _dm_dots(array, scale="auto"):
    """Compact transparent SVG: filled positives, outlined negatives, tiny zeros."""
    array = _df_value(array)
    if not array.size:
        return _DFHTML('<span title="Empty array">∅</span>')
    magnitude = _df_np.abs(array)
    finite = magnitude[_df_np.isfinite(magnitude)]
    reference = (float(finite.max()) if finite.size and finite.max() else 1.0) if scale == "auto" else float(scale)
    if array.ndim < 2:
        planes = [((), array.reshape(1, -1))]
    else:
        frames = list(_df_np.ndindex(array.shape[:-2]))
        if len(frames) > 64:
            raise ValueError("Too many panels")
        planes = [(frame, array[frame] if frame else array) for frame in frames]
    rows, cols = planes[0][1].shape
    width = cols * 24 + 16
    plane_height = rows * 24 + 16
    height = len(planes) * plane_height + (len(planes)-1) * 12
    shape = " × ".join(map(str, array.shape)) or "scalar"
    description = _df_html.escape(f"Shape {shape}. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude {reference:g}.", quote=True)
    marks = []
    for panel, (frame, plane) in enumerate(planes):
        for row in range(rows):
            for col in range(cols):
                value = plane[row, col]
                mag = float(abs(value))
                x, y = 20 + col * 24, panel * (plane_height+12) + 20 + row * 24
                index = frame + (() if array.ndim == 0 else (col,) if array.ndim == 1 else (row,col))
                title = _df_html.escape(f"{index}: {_df_number(value)}")
                if not _df_math.isfinite(mag):
                    marks.append(f'<text x="{x}" y="{y+4}" text-anchor="middle" fill="black"><title>{title}</title>×</text>')
                    continue
                radius = 1 if mag == 0 else 8 * _df_math.sqrt(min(mag/reference,1))
                negative = value.real < 0 and not (_df_np.iscomplexobj(value) and value.imag)
                fill = 'none' if negative else 'black'
                stroke = 'black' if negative else 'none'
                marks.append(f'<circle cx="{x}" cy="{y}" r="{radius:.4f}" fill="{fill}" stroke="{stroke}" stroke-width="1"><title>{title}</title></circle>')
    return _DFHTML(f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{description}" width="{width}" height="{height}" viewBox="0 0 {width} {height}" style="display:block;max-width:100%;height:auto;background:transparent"><title>{description}</title>{"".join(marks)}</svg>')


"""Browser animation support embedded in the BPL Dots setup cell."""
import json as _da_json


_DA_PAGE = r'''<!doctype html><meta charset="utf-8">
<style>
body{margin:0;background:transparent;color:black;font:12px system-ui}
canvas{display:block;max-width:100%;height:auto}
nav{display:flex;align-items:center;gap:12px;margin:4px 8px}
button{border:0;padding:2px 0;background:transparent;color:#555;font:inherit;font-size:11px;cursor:pointer}
button:disabled{opacity:.4;cursor:default}button:hover{color:black;text-decoration:underline}
output{color:#666}pre{margin:4px 8px;white-space:pre-wrap}
</style>
<pre id="numbers" hidden></pre><canvas aria-label="Animated BPL dots"></canvas>
<nav><button id="play" disabled>Pause</button><button id="step" disabled>Step</button>
<button id="restart" disabled>Restart</button><output>Loading BPL…</output></nav>
<script type="module">
const config = __CONFIG__;
const canvas = document.querySelector('canvas'), ctx = canvas.getContext('2d');
const status = document.querySelector('output'), numbers = document.querySelector('#numbers');
const play = document.querySelector('#play'), step = document.querySelector('#step');
const restart = document.querySelector('#restart');
let worker, workerURL, pending, timer, busy=false, running=true, tick=0, scale=null, latest, restartRequested=false;
const workerSource = `
import init, {Session, configure} from 'https://cdn.jsdelivr.net/npm/basedpl@0.1.82/basedpl_wasm.js';
await init();
configure('https://cdn.jsdelivr.net/npm/basedpl@0.1.82/', () => {});
let session = new Session();
onmessage = ({data}) => {
  try {
    if (data.reset) { session.free(); session = new Session(); }
    session.bind({t: data.t});
    const value = session.eval(data.code);
    postMessage({value});
  } catch (e) { postMessage({error:e.display ?? e.message ?? String(e)}); }
};
postMessage({ready:true});`;
function fit() {
  if (window.frameElement) window.frameElement.style.height = (Math.ceil(document.body.getBoundingClientRect().height)+4)+'px';
}
function fail(error) {
  running=false; clearTimeout(timer); worker?.terminate();
  if(workerURL) URL.revokeObjectURL(workerURL);
  play.disabled=step.disabled=true; restart.disabled=false;
  status.textContent=String(error); fit();
}
function numeric(value) {
  if(typeof value==='number' || typeof value==='boolean') return {shape:[],data:[Number(value)]};
  if(!value || !Array.isArray(value.shape) || !value.data) throw Error('Use a rectangular real numeric array as the last expression.');
  const data=Array.from(value.data);
  if(data.some(x=>typeof x!=='number' && typeof x!=='boolean')) throw Error('Animated dots support real numbers and Booleans.');
  return {shape:value.shape,data:data.map(Number)};
}
function draw(value) {
  const a=numeric(value), {shape,data}=a, rank=shape.length;
  if(data.length>4096) throw Error('Animated dots support up to 4096 values.');
  const rows=rank<2?1:shape[rank-2], cols=rank===0?1:shape[rank-1];
  const panels=rank<3?1:shape.slice(0,-2).reduce((a,b)=>a*b,1);
  if(panels>64) throw Error('Animated dots support up to 64 grids.');
  const width=Math.max(40,cols*24+16), ph=rows*24+16, height=Math.max(40,panels*ph+(panels-1)*12);
  const dpr=devicePixelRatio||1;
  canvas.width=width*dpr; canvas.height=height*dpr; canvas.style.width=width+'px';
  canvas.style.height=height+'px'; ctx.scale(dpr,dpr);
  if(scale===null) scale=config.scale==='auto' ? Math.max(...data.filter(Number.isFinite).map(Math.abs),0)||1 : config.scale;
  ctx.fillStyle=ctx.strokeStyle='black';ctx.lineWidth=1;
  data.forEach((v,i)=>{
    const panel=Math.floor(i/(rows*cols)), local=i%(rows*cols);
    const x=20+(local%cols)*24, y=20+Math.floor(local/cols)*24+panel*(ph+12);
    if(!Number.isFinite(v)){ctx.fillText('×',x-3,y+4);return;}
    const r=v===0?1:8*Math.sqrt(Math.min(Math.abs(v)/scale,1));
    ctx.beginPath();ctx.arc(x,y,r,0,Math.PI*2);v<0?ctx.stroke():ctx.fill();
  });
  if(!data.length)ctx.fillText('∅',12,24);
  canvas.setAttribute('aria-label',`BPL dots, shape ${shape.join(' by ')||'scalar'}, t ${tick}`);
  latest={...a,rows,cols,ph,width,height};
  numbers.hidden=!config.numbers;
  if(config.numbers)numbers.textContent=rank<2?data.join(' '):Array.from({length:panels},(_,p)=>Array.from({length:rows},(_,r)=>data.slice((p*rows+r)*cols,(p*rows+r+1)*cols).join(' ')).join('\n')).join('\n\n');
  status.textContent='t = '+tick; fit();
}
canvas.onmousemove=e=>{
  if(!latest)return;
  const {data,rows,cols,ph,width,height}=latest,b=canvas.getBoundingClientRect();
  const x=(e.clientX-b.left)*width/b.width,y=(e.clientY-b.top)*height/b.height;
  const panel=Math.floor(y/(ph+12)), row=Math.round((y-panel*(ph+12)-20)/24), col=Math.round((x-20)/24);
  canvas.title=row>=0&&row<rows&&col>=0&&col<cols?String(data[(panel*rows+row)*cols+col]??''):'';
};
function ask(payload) {
  return new Promise((resolve,reject)=>{
    const timeout=setTimeout(()=>{pending=null;reject(Error('BPL frame timed out. Try a smaller expression.'));},10000);
    pending={resolve:v=>{clearTimeout(timeout);resolve(v);},reject:e=>{clearTimeout(timeout);reject(e);}};
    worker.postMessage(payload);
  });
}
async function frame(reset=false) {
  if(busy)return;
  busy=true;
  try {const value=await ask({code:config.code,t:tick,reset});draw(value);tick++;}
  catch(e){fail(e.message??e);}finally{busy=false;}
  if(restartRequested){restartRequested=false;restart.onclick();return;}
  if(running)schedule();
}
function schedule(){timer=setTimeout(()=>{if(!running)return;if(document.hidden)schedule();else frame();},1000/config.fps);}
play.onclick=()=>{running=!running;play.textContent=running?'Pause':'Play';clearTimeout(timer);if(running&&!busy)frame();};
step.onclick=()=>{running=false;play.textContent='Play';clearTimeout(timer);if(!busy)frame();};
restart.onclick=()=>{if(busy){restartRequested=true;return;}clearTimeout(timer);worker?.terminate();if(workerURL)URL.revokeObjectURL(workerURL);pending=null;busy=false;tick=0;scale=null;running=true;play.textContent='Pause';start();};
function start() {
  play.disabled=step.disabled=restart.disabled=true;status.textContent='Loading BPL…';
  workerURL=URL.createObjectURL(new Blob([workerSource],{type:'text/javascript'}));
  worker=new Worker(workerURL,{type:'module'});
  const loading=setTimeout(()=>fail('BPL could not load. Check your internet connection, then Restart.'),30000);
  worker.onmessage=({data})=>{
    if(data.ready){clearTimeout(loading);play.disabled=step.disabled=restart.disabled=false;frame();return;}
    const p=pending;pending=null;
    if(data.error)p?.reject(Error(data.error));else p?.resolve(data.value);
  };
  worker.onerror=e=>{clearTimeout(loading);if(pending){pending.reject(Error(e.message));pending=null;}else fail(e.message||'BPL could not load.');};
}
addEventListener('pagehide',()=>{running=false;clearTimeout(timer);worker?.terminate();if(workerURL)URL.revokeObjectURL(workerURL);});
start();
</script>'''


def _dm_has_time(node):
    """Detect the variable t, not text, comments or a name such as total."""
    return (node.kind == 'Name' and node.text == 't') or any(_dm_has_time(child) for child in node.children)


def _dm_animated(code, options):
    config = _da_json.dumps(dict(code=code, scale=options.scale,
                                fps=options.fps, numbers=options.numbers),
                           ensure_ascii=False).replace('</', r'<\/')
    page = _DA_PAGE.replace('__CONFIG__', config)
    return {'text/html': '<iframe title="Animated BPL dots" srcdoc="' +
                   _df_html.escape(page, quote=True) +
                   '" style="display:block;width:100%;height:170px;border:0;background:transparent"></iframe>'}


def install_dot_magic(shell=None):
    """Extend this kernel's magic only. Running setup again does not stack wrappers."""
    shell = shell or _dm_get_ipython()
    if shell is None:
        raise RuntimeError("Run setup in a notebook kernel.")
    original = shell.find_cell_magic("bpl")
    if original is None:
        _dm_notebooks.create_magic(shell)
        original = shell.find_cell_magic("bpl")
    original = getattr(original, "_bpl_dot_original", original)

    def bpl_with_dots(line, cell=None):
        # The standard magic remains the complete implementation without --dots.
        if not {"--dots", "--adots"}.intersection(line.split()):
            return original(line, cell)
        parser = _DotMagicParser(prog="%%bpl", add_help=False)
        parser.add_argument("--dots", action="store_true")
        parser.add_argument("--adots", action="store_true")
        parser.add_argument("--fps", type=float, default=4)
        parser.add_argument("--numbers", action="store_true")
        parser.add_argument("--scale", default="auto")
        if cell is None:
            raise _DMUsageError("Use a cell headed %%bpl --dots, then put BPL below it.")
        options = parser.parse_args(_dm_shlex.split(line))
        if not _df_math.isfinite(options.fps) or not 0 < options.fps <= 60:
            raise _DMUsageError("--fps needs a number above zero and at most 60.")
        scale = options.scale
        if scale != "auto":
            try:
                scale = float(scale)
            except ValueError:
                raise _DMUsageError("--scale needs a positive number or auto.") from None
            if not _df_math.isfinite(scale) or scale <= 0:
                raise _DMUsageError("--scale needs a positive number or auto.")
        code = cell.rstrip()
        if code.endswith(";"):
            return original("", cell)
        tree = _dm_parse(code)
        # Preserve native handling and errors for commands and unparsable input.
        if any(node.kind == "Error" for node in tree.children):
            return original("", cell)
        if options.adots and _dm_has_time(tree):
            options.scale = scale
            _dm_display(_dm_animated(code, options), raw=True)
            return None
        for node in tree.children:
            if node.kind != "Statement":
                continue
            # Ask BPL's parser for complete statements, never split on newlines:
            # multiline functions, strings, comments and diamonds stay intact.
            # Whitespace padding keeps original error line and column positions.
            start, end = node.span
            prefix = "".join("\n" if char == "\n" else " " for char in code[:start])
            statement = prefix + code[start:end]
            events = []
            try:
                result = _dm_bp.bpl._eval(statement, {}, events.append, echo=True)
            except Exception:
                for event in events:
                    _dm_notebooks.display_event(event)
                raise
            implicit = [event for event in events if event["kind"] == "display"]
            visual = None
            if len(implicit) == 1 and isinstance(result.value, _dm_bp.Array):
                try:
                    array = _df_np.asarray(result.value.np)
                    # Do not turn strings containing digits or enclosed arrays
                    # into simple numeric arrays as a side effect of display.
                    if array.dtype.kind in "biufc" and tuple(array.shape) == tuple(result.value.shape):
                        visual = _dm_dots(array, scale)
                except (TypeError, ValueError, OverflowError):
                    pass  # Text, functions, large/ragged arrays keep native output.
            for event in events:
                if visual is not None and event["kind"] == "display":
                    if options.numbers:
                        _dm_notebooks.display_event(event)
                    _dm_display(visual)
                else:
                    _dm_notebooks.display_event(event)
        return None

    bpl_with_dots._bpl_dot_original = original
    shell.register_magic_function(bpl_with_dots, "line_cell", "bpl")
    return bpl_with_dots


if _dm_get_ipython() is not None:
    install_dot_magic()
    print("BPL ready: %%bpl --dots or %%bpl --adots. Optional: --numbers, --scale 1, --fps 4.")


BPL ready: %%bpl --dots or %%bpl --adots. Optional: --numbers, --scale 1, --fps 4.


## Normal BPL

In [ ]:
%%bpl
dotMatrix←2 3⍴↦6
dotMatrix

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────┐
↓1 2 3│
│4 5 6│
└~────┘

Plain `%%bpl` displays the usual numbers. This also stores `dotMatrix` for later examples.

## Dot Output

Add `--dots` to show dots, or `--dots --numbers` to show numbers followed by dots. Dot area shows magnitude, and positions follow the array shape. Positive values are filled, negatives are hollow, and zero is a tiny point. Hover for the exact value.

In [ ]:
%%bpl --dots
2 3⍴↦6

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 2 × 3. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 6." width="88" height="64" viewBox="0 0 88 64" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 2 × 3. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 6.</title><circle cx="20" cy="20" r="3.2660" fill="black" stroke="none" stroke-width="1"><title>(0, 0): 1</title></circle><circle cx="44" cy="20" r="4.6188" fill="black" stroke="none" stroke-width="1"><title>(0, 1): 2</title></circle><circle cx="68" cy="20" r="5.6569" fill="black" stroke="none" stroke-width="1"><title>(0, 2): 3</title></circle><circle cx="20" cy="44" r="6.5320" fill="black" stroke="none" stroke-width="1"><title>(1, 0): 4</title></circle><circle cx="44" cy="44" r="7.3030" fill="black" stroke="none" stroke-width="1"><title>(1, 1): 5</title></circle><circle cx="68" cy="44" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(1, 2): 6</title></circle></svg>)

Two rows and three columns. The largest magnitude fills a dot; smaller magnitudes get proportionally smaller areas.

In [ ]:
%%bpl --dots --numbers
dotMatrix

┌→────┐
↓1 2 3│
│4 5 6│
└~────┘

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 2 × 3. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 6." width="88" height="64" viewBox="0 0 88 64" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 2 × 3. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 6.</title><circle cx="20" cy="20" r="3.2660" fill="black" stroke="none" stroke-width="1"><title>(0, 0): 1</title></circle><circle cx="44" cy="20" r="4.6188" fill="black" stroke="none" stroke-width="1"><title>(0, 1): 2</title></circle><circle cx="68" cy="20" r="5.6569" fill="black" stroke="none" stroke-width="1"><title>(0, 2): 3</title></circle><circle cx="20" cy="44" r="6.5320" fill="black" stroke="none" stroke-width="1"><title>(1, 0): 4</title></circle><circle cx="44" cy="44" r="7.3030" fill="black" stroke="none" stroke-width="1"><title>(1, 1): 5</title></circle><circle cx="68" cy="44" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(1, 2): 6</title></circle></svg>)

The same result, shown as both numbers and dots. BPL evaluates the expression once.

In [ ]:
%%bpl --dots
⍉4 4⍴(⍳16)÷15

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 4 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1." width="112" height="112" viewBox="0 0 112 112" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 4 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1.</title><circle cx="20" cy="20" r="1.0000" fill="black" stroke="none" stroke-width="1"><title>(0, 0): 0</title></circle><circle cx="44" cy="20" r="4.1312" fill="black" stroke="none" stroke-width="1"><title>(0, 1): 0.26666666666666666</title></circle><circle cx="68" cy="20" r="5.8424" fill="black" stroke="none" stroke-width="1"><title>(0, 2): 0.53333333333333333</title></circle><circle cx="92" cy="20" r="7.1554" fill="black" stroke="none" stroke-width="1"><title>(0, 3): 0.80000000000000004</title></circle><circle cx="20" cy="44" r="2.0656" fill="black" stroke="none" stroke-width="1"><title>(1, 0): 0.066666666666666666</title></circle><circle cx="44" cy="44" r="4.6188" fill="black" stroke="none" stroke-width="1"><title>(1, 1): 0.33333333333333331</title></circle><circle cx="68" cy="44" r="6.1968" fill="black" stroke="none" stroke-width="1"><title>(1, 2): 0.59999999999999998</title></circle><circle cx="92" cy="44" r="7.4476" fill="black" stroke="none" stroke-width="1"><title>(1, 3): 0.8666666666666667</title></circle><circle cx="20" cy="68" r="2.9212" fill="black" stroke="none" stroke-width="1"><title>(2, 0): 0.13333333333333333</title></circle><circle cx="44" cy="68" r="5.0596" fill="black" stroke="none" stroke-width="1"><title>(2, 1): 0.40000000000000002</title></circle><circle cx="68" cy="68" r="6.5320" fill="black" stroke="none" stroke-width="1"><title>(2, 2): 0.66666666666666663</title></circle><circle cx="92" cy="68" r="7.7287" fill="black" stroke="none" stroke-width="1"><title>(2, 3): 0.93333333333333335</title></circle><circle cx="20" cy="92" r="3.5777" fill="black" stroke="none" stroke-width="1"><title>(3, 0): 0.20000000000000001</title></circle><circle cx="44" cy="92" r="5.4650" fill="black" stroke="none" stroke-width="1"><title>(3, 1): 0.46666666666666667</title></circle><circle cx="68" cy="92" r="6.8508" fill="black" stroke="none" stroke-width="1"><title>(3, 2): 0.73333333333333328</title></circle><circle cx="92" cy="92" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(3, 3): 1</title></circle></svg>)

Transpose swaps rows and columns, so the values increase across the columns of this `4×4` grid.

In [ ]:
%%bpl --dots
dotMatrix
⍉dotMatrix

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 2 × 3. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 6." width="88" height="64" viewBox="0 0 88 64" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 2 × 3. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 6.</title><circle cx="20" cy="20" r="3.2660" fill="black" stroke="none" stroke-width="1"><title>(0, 0): 1</title></circle><circle cx="44" cy="20" r="4.6188" fill="black" stroke="none" stroke-width="1"><title>(0, 1): 2</title></circle><circle cx="68" cy="20" r="5.6569" fill="black" stroke="none" stroke-width="1"><title>(0, 2): 3</title></circle><circle cx="20" cy="44" r="6.5320" fill="black" stroke="none" stroke-width="1"><title>(1, 0): 4</title></circle><circle cx="44" cy="44" r="7.3030" fill="black" stroke="none" stroke-width="1"><title>(1, 1): 5</title></circle><circle cx="68" cy="44" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(1, 2): 6</title></circle></svg>)

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 3 × 2. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 6." width="64" height="88" viewBox="0 0 64 88" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 3 × 2. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 6.</title><circle cx="20" cy="20" r="3.2660" fill="black" stroke="none" stroke-width="1"><title>(0, 0): 1</title></circle><circle cx="44" cy="20" r="6.5320" fill="black" stroke="none" stroke-width="1"><title>(0, 1): 4</title></circle><circle cx="20" cy="44" r="4.6188" fill="black" stroke="none" stroke-width="1"><title>(1, 0): 2</title></circle><circle cx="44" cy="44" r="7.3030" fill="black" stroke="none" stroke-width="1"><title>(1, 1): 5</title></circle><circle cx="20" cy="68" r="5.6569" fill="black" stroke="none" stroke-width="1"><title>(2, 0): 3</title></circle><circle cx="44" cy="68" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(2, 1): 6</title></circle></svg>)

Multiple expressions produce separate outputs: the `2×3` matrix, then its `3×2` transpose.

In [ ]:
%%bpl --dots --scale 1
⍳16
(⍳16)÷15
4 4⍴(⍳16)÷15
⍉4 4⍴(⍳16)÷15

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 16. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1." width="400" height="40" viewBox="0 0 400 40" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 16. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1.</title><circle cx="20" cy="20" r="1.0000" fill="black" stroke="none" stroke-width="1"><title>(0,): 0</title></circle><circle cx="44" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(1,): 1</title></circle><circle cx="68" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(2,): 2</title></circle><circle cx="92" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(3,): 3</title></circle><circle cx="116" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(4,): 4</title></circle><circle cx="140" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(5,): 5</title></circle><circle cx="164" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(6,): 6</title></circle><circle cx="188" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(7,): 7</title></circle><circle cx="212" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(8,): 8</title></circle><circle cx="236" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(9,): 9</title></circle><circle cx="260" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(10,): 10</title></circle><circle cx="284" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(11,): 11</title></circle><circle cx="308" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(12,): 12</title></circle><circle cx="332" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(13,): 13</title></circle><circle cx="356" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(14,): 14</title></circle><circle cx="380" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(15,): 15</title></circle></svg>)

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 16. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1." width="400" height="40" viewBox="0 0 400 40" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 16. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1.</title><circle cx="20" cy="20" r="1.0000" fill="black" stroke="none" stroke-width="1"><title>(0,): 0</title></circle><circle cx="44" cy="20" r="2.0656" fill="black" stroke="none" stroke-width="1"><title>(1,): 0.066666666666666666</title></circle><circle cx="68" cy="20" r="2.9212" fill="black" stroke="none" stroke-width="1"><title>(2,): 0.13333333333333333</title></circle><circle cx="92" cy="20" r="3.5777" fill="black" stroke="none" stroke-width="1"><title>(3,): 0.20000000000000001</title></circle><circle cx="116" cy="20" r="4.1312" fill="black" stroke="none" stroke-width="1"><title>(4,): 0.26666666666666666</title></circle><circle cx="140" cy="20" r="4.6188" fill="black" stroke="none" stroke-width="1"><title>(5,): 0.33333333333333331</title></circle><circle cx="164" cy="20" r="5.0596" fill="black" stroke="none" stroke-width="1"><title>(6,): 0.40000000000000002</title></circle><circle cx="188" cy="20" r="5.4650" fill="black" stroke="none" stroke-width="1"><title>(7,): 0.46666666666666667</title></circle><circle cx="212" cy="20" r="5.8424" fill="black" stroke="none" stroke-width="1"><title>(8,): 0.53333333333333333</title></circle><circle cx="236" cy="20" r="6.1968" fill="black" stroke="none" stroke-width="1"><title>(9,): 0.59999999999999998</title></circle><circle cx="260" cy="20" r="6.5320" fill="black" stroke="none" stroke-width="1"><title>(10,): 0.66666666666666663</title></circle><circle cx="284" cy="20" r="6.8508" fill="black" stroke="none" stroke-width="1"><title>(11,): 0.73333333333333328</title></circle><circle cx="308" cy="20" r="7.1554" fill="black" stroke="none" stroke-width="1"><title>(12,): 0.80000000000000004</title></circle><circle cx="332" cy="20" r="7.4476" fill="black" stroke="none" stroke-width="1"><title>(13,): 0.8666666666666667</title></circle><circle cx="356" cy="20" r="7.7287" fill="black" stroke="none" stroke-width="1"><title>(14,): 0.93333333333333335</title></circle><circle cx="380" cy="20" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(15,): 1</title></circle></svg>)

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 4 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1." width="112" height="112" viewBox="0 0 112 112" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 4 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1.</title><circle cx="20" cy="20" r="1.0000" fill="black" stroke="none" stroke-width="1"><title>(0, 0): 0</title></circle><circle cx="44" cy="20" r="2.0656" fill="black" stroke="none" stroke-width="1"><title>(0, 1): 0.066666666666666666</title></circle><circle cx="68" cy="20" r="2.9212" fill="black" stroke="none" stroke-width="1"><title>(0, 2): 0.13333333333333333</title></circle><circle cx="92" cy="20" r="3.5777" fill="black" stroke="none" stroke-width="1"><title>(0, 3): 0.20000000000000001</title></circle><circle cx="20" cy="44" r="4.1312" fill="black" stroke="none" stroke-width="1"><title>(1, 0): 0.26666666666666666</title></circle><circle cx="44" cy="44" r="4.6188" fill="black" stroke="none" stroke-width="1"><title>(1, 1): 0.33333333333333331</title></circle><circle cx="68" cy="44" r="5.0596" fill="black" stroke="none" stroke-width="1"><title>(1, 2): 0.40000000000000002</title></circle><circle cx="92" cy="44" r="5.4650" fill="black" stroke="none" stroke-width="1"><title>(1, 3): 0.46666666666666667</title></circle><circle cx="20" cy="68" r="5.8424" fill="black" stroke="none" stroke-width="1"><title>(2, 0): 0.53333333333333333</title></circle><circle cx="44" cy="68" r="6.1968" fill="black" stroke="none" stroke-width="1"><title>(2, 1): 0.59999999999999998</title></circle><circle cx="68" cy="68" r="6.5320" fill="black" stroke="none" stroke-width="1"><title>(2, 2): 0.66666666666666663</title></circle><circle cx="92" cy="68" r="6.8508" fill="black" stroke="none" stroke-width="1"><title>(2, 3): 0.73333333333333328</title></circle><circle cx="20" cy="92" r="7.1554" fill="black" stroke="none" stroke-width="1"><title>(3, 0): 0.80000000000000004</title></circle><circle cx="44" cy="92" r="7.4476" fill="black" stroke="none" stroke-width="1"><title>(3, 1): 0.8666666666666667</title></circle><circle cx="68" cy="92" r="7.7287" fill="black" stroke="none" stroke-width="1"><title>(3, 2): 0.93333333333333335</title></circle><circle cx="92" cy="92" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(3, 3): 1</title></circle></svg>)

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 4 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1." width="112" height="112" viewBox="0 0 112 112" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 4 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1.</title><circle cx="20" cy="20" r="1.0000" fill="black" stroke="none" stroke-width="1"><title>(0, 0): 0</title></circle><circle cx="44" cy="20" r="4.1312" fill="black" stroke="none" stroke-width="1"><title>(0, 1): 0.26666666666666666</title></circle><circle cx="68" cy="20" r="5.8424" fill="black" stroke="none" stroke-width="1"><title>(0, 2): 0.53333333333333333</title></circle><circle cx="92" cy="20" r="7.1554" fill="black" stroke="none" stroke-width="1"><title>(0, 3): 0.80000000000000004</title></circle><circle cx="20" cy="44" r="2.0656" fill="black" stroke="none" stroke-width="1"><title>(1, 0): 0.066666666666666666</title></circle><circle cx="44" cy="44" r="4.6188" fill="black" stroke="none" stroke-width="1"><title>(1, 1): 0.33333333333333331</title></circle><circle cx="68" cy="44" r="6.1968" fill="black" stroke="none" stroke-width="1"><title>(1, 2): 0.59999999999999998</title></circle><circle cx="92" cy="44" r="7.4476" fill="black" stroke="none" stroke-width="1"><title>(1, 3): 0.8666666666666667</title></circle><circle cx="20" cy="68" r="2.9212" fill="black" stroke="none" stroke-width="1"><title>(2, 0): 0.13333333333333333</title></circle><circle cx="44" cy="68" r="5.0596" fill="black" stroke="none" stroke-width="1"><title>(2, 1): 0.40000000000000002</title></circle><circle cx="68" cy="68" r="6.5320" fill="black" stroke="none" stroke-width="1"><title>(2, 2): 0.66666666666666663</title></circle><circle cx="92" cy="68" r="7.7287" fill="black" stroke="none" stroke-width="1"><title>(2, 3): 0.93333333333333335</title></circle><circle cx="20" cy="92" r="3.5777" fill="black" stroke="none" stroke-width="1"><title>(3, 0): 0.20000000000000001</title></circle><circle cx="44" cy="92" r="5.4650" fill="black" stroke="none" stroke-width="1"><title>(3, 1): 0.46666666666666667</title></circle><circle cx="68" cy="92" r="6.8508" fill="black" stroke="none" stroke-width="1"><title>(3, 2): 0.73333333333333328</title></circle><circle cx="92" cy="92" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(3, 3): 1</title></circle></svg>)

`--scale 1` keeps the same size scale across these four expressions. Values above 1 are capped at a full dot. Division produces a gradual change in size; reshape and transpose rearrange the positions. Without this option, each output scales to its own largest magnitude.

In [ ]:
%%bpl --dots
((4 5⍴⍳20)-9.5)÷9.5

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 4 × 5. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1." width="136" height="112" viewBox="0 0 136 112" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 4 × 5. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1.</title><circle cx="20" cy="20" r="8.0000" fill="none" stroke="black" stroke-width="1"><title>(0, 0): ¯1</title></circle><circle cx="44" cy="20" r="7.5672" fill="none" stroke="black" stroke-width="1"><title>(0, 1): ¯0.89473684210526316</title></circle><circle cx="68" cy="20" r="7.1082" fill="none" stroke="black" stroke-width="1"><title>(0, 2): ¯0.78947368421052633</title></circle><circle cx="92" cy="20" r="6.6174" fill="none" stroke="black" stroke-width="1"><title>(0, 3): ¯0.68421052631578949</title></circle><circle cx="116" cy="20" r="6.0871" fill="none" stroke="black" stroke-width="1"><title>(0, 4): ¯0.57894736842105265</title></circle><circle cx="20" cy="44" r="5.5060" fill="none" stroke="black" stroke-width="1"><title>(1, 0): ¯0.47368421052631576</title></circle><circle cx="44" cy="44" r="4.8558" fill="none" stroke="black" stroke-width="1"><title>(1, 1): ¯0.36842105263157893</title></circle><circle cx="68" cy="44" r="4.1039" fill="none" stroke="black" stroke-width="1"><title>(1, 2): ¯0.26315789473684209</title></circle><circle cx="92" cy="44" r="3.1789" fill="none" stroke="black" stroke-width="1"><title>(1, 3): ¯0.15789473684210525</title></circle><circle cx="116" cy="44" r="1.8353" fill="none" stroke="black" stroke-width="1"><title>(1, 4): ¯0.052631578947368418</title></circle><circle cx="20" cy="68" r="1.8353" fill="black" stroke="none" stroke-width="1"><title>(2, 0): 0.052631578947368418</title></circle><circle cx="44" cy="68" r="3.1789" fill="black" stroke="none" stroke-width="1"><title>(2, 1): 0.15789473684210525</title></circle><circle cx="68" cy="68" r="4.1039" fill="black" stroke="none" stroke-width="1"><title>(2, 2): 0.26315789473684209</title></circle><circle cx="92" cy="68" r="4.8558" fill="black" stroke="none" stroke-width="1"><title>(2, 3): 0.36842105263157893</title></circle><circle cx="116" cy="68" r="5.5060" fill="black" stroke="none" stroke-width="1"><title>(2, 4): 0.47368421052631576</title></circle><circle cx="20" cy="92" r="6.0871" fill="black" stroke="none" stroke-width="1"><title>(3, 0): 0.57894736842105265</title></circle><circle cx="44" cy="92" r="6.6174" fill="black" stroke="none" stroke-width="1"><title>(3, 1): 0.68421052631578949</title></circle><circle cx="68" cy="92" r="7.1082" fill="black" stroke="none" stroke-width="1"><title>(3, 2): 0.78947368421052633</title></circle><circle cx="92" cy="92" r="7.5672" fill="black" stroke="none" stroke-width="1"><title>(3, 3): 0.89473684210526316</title></circle><circle cx="116" cy="92" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(3, 4): 1</title></circle></svg>)

In [ ]:
%%bpl --dots
4 4⍴-⍳16

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 4 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 15." width="112" height="112" viewBox="0 0 112 112" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 4 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 15.</title><circle cx="20" cy="20" r="1.0000" fill="black" stroke="none" stroke-width="1"><title>(0, 0): ¯0</title></circle><circle cx="44" cy="20" r="2.0656" fill="none" stroke="black" stroke-width="1"><title>(0, 1): ¯1</title></circle><circle cx="68" cy="20" r="2.9212" fill="none" stroke="black" stroke-width="1"><title>(0, 2): ¯2</title></circle><circle cx="92" cy="20" r="3.5777" fill="none" stroke="black" stroke-width="1"><title>(0, 3): ¯3</title></circle><circle cx="20" cy="44" r="4.1312" fill="none" stroke="black" stroke-width="1"><title>(1, 0): ¯4</title></circle><circle cx="44" cy="44" r="4.6188" fill="none" stroke="black" stroke-width="1"><title>(1, 1): ¯5</title></circle><circle cx="68" cy="44" r="5.0596" fill="none" stroke="black" stroke-width="1"><title>(1, 2): ¯6</title></circle><circle cx="92" cy="44" r="5.4650" fill="none" stroke="black" stroke-width="1"><title>(1, 3): ¯7</title></circle><circle cx="20" cy="68" r="5.8424" fill="none" stroke="black" stroke-width="1"><title>(2, 0): ¯8</title></circle><circle cx="44" cy="68" r="6.1968" fill="none" stroke="black" stroke-width="1"><title>(2, 1): ¯9</title></circle><circle cx="68" cy="68" r="6.5320" fill="none" stroke="black" stroke-width="1"><title>(2, 2): ¯10</title></circle><circle cx="92" cy="68" r="6.8508" fill="none" stroke="black" stroke-width="1"><title>(2, 3): ¯11</title></circle><circle cx="20" cy="92" r="7.1554" fill="none" stroke="black" stroke-width="1"><title>(3, 0): ¯12</title></circle><circle cx="44" cy="92" r="7.4476" fill="none" stroke="black" stroke-width="1"><title>(3, 1): ¯13</title></circle><circle cx="68" cy="92" r="7.7287" fill="none" stroke="black" stroke-width="1"><title>(3, 2): ¯14</title></circle><circle cx="92" cy="92" r="8.0000" fill="none" stroke="black" stroke-width="1"><title>(3, 3): ¯15</title></circle></svg>)

This `4×5` array runs from ¯1 to 1. Hollow circles show negative values; filled circles show positive values.

In [ ]:
%%bpl --dots
2 3 4⍴(⍳24)÷23

HTML(<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Shape 2 × 3 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1." width="112" height="188" viewBox="0 0 112 188" style="display:block;max-width:100%;height:auto;background:transparent"><title>Shape 2 × 3 × 4. Filled: positive; outlined: negative; tiny dot: zero. Full dot magnitude 1.</title><circle cx="20" cy="20" r="1.0000" fill="black" stroke="none" stroke-width="1"><title>(0, 0, 0): 0</title></circle><circle cx="44" cy="20" r="1.6681" fill="black" stroke="none" stroke-width="1"><title>(0, 0, 1): 0.043478260869565216</title></circle><circle cx="68" cy="20" r="2.3591" fill="black" stroke="none" stroke-width="1"><title>(0, 0, 2): 0.086956521739130432</title></circle><circle cx="92" cy="20" r="2.8893" fill="black" stroke="none" stroke-width="1"><title>(0, 0, 3): 0.13043478260869565</title></circle><circle cx="20" cy="44" r="3.3362" fill="black" stroke="none" stroke-width="1"><title>(0, 1, 0): 0.17391304347826086</title></circle><circle cx="44" cy="44" r="3.7300" fill="black" stroke="none" stroke-width="1"><title>(0, 1, 1): 0.21739130434782608</title></circle><circle cx="68" cy="44" r="4.0860" fill="black" stroke="none" stroke-width="1"><title>(0, 1, 2): 0.2608695652173913</title></circle><circle cx="92" cy="44" r="4.4134" fill="black" stroke="none" stroke-width="1"><title>(0, 1, 3): 0.30434782608695654</title></circle><circle cx="20" cy="68" r="4.7181" fill="black" stroke="none" stroke-width="1"><title>(0, 2, 0): 0.34782608695652173</title></circle><circle cx="44" cy="68" r="5.0043" fill="black" stroke="none" stroke-width="1"><title>(0, 2, 1): 0.39130434782608697</title></circle><circle cx="68" cy="68" r="5.2750" fill="black" stroke="none" stroke-width="1"><title>(0, 2, 2): 0.43478260869565216</title></circle><circle cx="92" cy="68" r="5.5325" fill="black" stroke="none" stroke-width="1"><title>(0, 2, 3): 0.47826086956521741</title></circle><circle cx="20" cy="120" r="5.7785" fill="black" stroke="none" stroke-width="1"><title>(1, 0, 0): 0.52173913043478259</title></circle><circle cx="44" cy="120" r="6.0145" fill="black" stroke="none" stroke-width="1"><title>(1, 0, 1): 0.56521739130434778</title></circle><circle cx="68" cy="120" r="6.2415" fill="black" stroke="none" stroke-width="1"><title>(1, 0, 2): 0.60869565217391308</title></circle><circle cx="92" cy="120" r="6.4606" fill="black" stroke="none" stroke-width="1"><title>(1, 0, 3): 0.65217391304347827</title></circle><circle cx="20" cy="144" r="6.6725" fill="black" stroke="none" stroke-width="1"><title>(1, 1, 0): 0.69565217391304346</title></circle><circle cx="44" cy="144" r="6.8778" fill="black" stroke="none" stroke-width="1"><title>(1, 1, 1): 0.73913043478260865</title></circle><circle cx="68" cy="144" r="7.0772" fill="black" stroke="none" stroke-width="1"><title>(1, 1, 2): 0.78260869565217395</title></circle><circle cx="92" cy="144" r="7.2711" fill="black" stroke="none" stroke-width="1"><title>(1, 1, 3): 0.82608695652173914</title></circle><circle cx="20" cy="168" r="7.4600" fill="black" stroke="none" stroke-width="1"><title>(1, 2, 0): 0.86956521739130432</title></circle><circle cx="44" cy="168" r="7.6443" fill="black" stroke="none" stroke-width="1"><title>(1, 2, 1): 0.91304347826086951</title></circle><circle cx="68" cy="168" r="7.8242" fill="black" stroke="none" stroke-width="1"><title>(1, 2, 2): 0.95652173913043481</title></circle><circle cx="92" cy="168" r="8.0000" fill="black" stroke="none" stroke-width="1"><title>(1, 2, 3): 1</title></circle></svg>)

Shape `2 3 4` appears as two grids, each with three rows and four columns.

## Normal Output Still Works

In [ ]:
%%bpl
+/dotMatrix

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

6 15

Leaving off `--dots` gives normal output again: the row sums are 6 and 15. The flag applies only to its own cell. Assignments stay silent, and a trailing semicolon suppresses output. Text, functions, nested arrays and explicit `⎕←` printing keep their normal display. This MVP draws up to 4096 values and 64 grids; larger results use normal output.

## Animated Dots

Use `%%bpl --adots` with the variable `t`. It starts at 0 and increases by 1 each frame. Pause holds the picture, Step advances once, and Restart begins again at 0. `--fps` sets frames per second; the default is 4.

In [ ]:
%%bpl --adots --numbers
4 4⍴(⍳16)=16|t

<iframe title="Animated BPL dots" srcdoc="<!doctype html><meta charset="utf-8">
<style>
body{margin:0;background:transparent;color:black;font:12px system-ui}
canvas{display:block;max-width:100%;height:auto}
nav{display:flex;align-items:center;gap:12px;margin:4px 8px}
button{border:0;padding:2px 0;background:transparent;color:#555;font:inherit;font-size:11px;cursor:pointer}
button:disabled{opacity:.4;cursor:default}button:hover{color:black;text-decoration:underline}
output{color:#666}pre{margin:4px 8px;white-space:pre-wrap}
</style>
<pre id="numbers" hidden></pre><canvas aria-label="Animated BPL dots"></canvas>
<nav><button id="play" disabled>Pause</button><button id="step" disabled>Step</button>
<button id="restart" disabled>Restart</button><output>Loading BPL…</output></nav>
<script type="module">
const config = {"code": "4 4⍴(⍳16)=16|t", "scale": "auto", "fps": 4, "numbers": true};
const canvas = document.querySelector('canvas'), ctx = canvas.getContext('2d');
const status = document.querySelector('output'), numbers = document.querySelector('#numbers');
const play = document.querySelector('#play'), step = document.querySelector('#step');
const restart = document.querySelector('#restart');
let worker, workerURL, pending, timer, busy=false, running=true, tick=0, scale=null, latest, restartRequested=false;
const workerSource = `
import init, {Session, configure} from 'https://cdn.jsdelivr.net/npm/basedpl@0.1.82/basedpl_wasm.js';
await init();
configure('https://cdn.jsdelivr.net/npm/basedpl@0.1.82/', () => {});
let session = new Session();
onmessage = ({data}) => {
 try {
 if (data.reset) { session.free(); session = new Session(); }
 session.bind({t: data.t});
 const value = session.eval(data.code);
 postMessage({value});
 } catch (e) { postMessage({error:e.display ?? e.message ?? String(e)}); }
};
postMessage({ready:true});`;
function fit() {
 if (window.frameElement) window.frameElement.style.height = (Math.ceil(document.body.getBoundingClientRect().height)+4)+'px';
}
function fail(error) {
 running=false; clearTimeout(timer); worker?.terminate();
 if(workerURL) URL.revokeObjectURL(workerURL);
 play.disabled=step.disabled=true; restart.disabled=false;
 status.textContent=String(error); fit();
}
function numeric(value) {
 if(typeof value==='number' || typeof value==='boolean') return {shape:[],data:[Number(value)]};
 if(!value || !Array.isArray(value.shape) || !value.data) throw Error('Use a rectangular real numeric array as the last expression.');
 const data=Array.from(value.data);
 if(data.some(x=>typeof x!=='number' && typeof x!=='boolean')) throw Error('Animated dots support real numbers and Booleans.');
 return {shape:value.shape,data:data.map(Number)};
}
function draw(value) {
 const a=numeric(value), {shape,data}=a, rank=shape.length;
 if(data.length>4096) throw Error('Animated dots support up to 4096 values.');
 const rows=rank<2?1:shape[rank-2], cols=rank===0?1:shape[rank-1];
 const panels=rank<3?1:shape.slice(0,-2).reduce((a,b)=>a*b,1);
 if(panels>64) throw Error('Animated dots support up to 64 grids.');
 const width=Math.max(40,cols*24+16), ph=rows*24+16, height=Math.max(40,panels*ph+(panels-1)*12);
 const dpr=devicePixelRatio||1;
 canvas.width=width*dpr; canvas.height=height*dpr; canvas.style.width=width+'px';
 canvas.style.height=height+'px'; ctx.scale(dpr,dpr);
 if(scale===null) scale=config.scale==='auto' ? Math.max(...data.filter(Number.isFinite).map(Math.abs),0)||1 : config.scale;
 ctx.fillStyle=ctx.strokeStyle='black';ctx.lineWidth=1;
 data.forEach((v,i)=>{
 const panel=Math.floor(i/(rows*cols)), local=i%(rows*cols);
 const x=20+(local%cols)*24, y=20+Math.floor(local/cols)*24+panel*(ph+12);
 if(!Number.isFinite(v)){ctx.fillText('×',x-3,y+4);return;}
 const r=v===0?1:8*Math.sqrt(Math.min(Math.abs(v)/scale,1));
 ctx.beginPath();ctx.arc(x,y,r,0,Math.PI*2);v<0?ctx.stroke():ctx.fill();
 });
 if(!data.length)ctx.fillText('∅',12,24);
 canvas.setAttribute('aria-label',`BPL dots, shape ${shape.join(' by ')||'scalar'},

`16|t` cycles through 0 to 15. Comparing that number with `⍳16` gives one true value, so one full dot moves through the grid. The picture repeats every 16 frames, while `t` keeps increasing.

In [ ]:
%%bpl --adots --fps 20 --scale 1
4 4⍴0.5+0.5×1○(((⍳16)÷3)+(t÷8))

<iframe title="Animated BPL dots" srcdoc="<!doctype html><meta charset="utf-8">
<style>
body{margin:0;background:transparent;color:black;font:12px system-ui}
canvas{display:block;max-width:100%;height:auto}
nav{display:flex;align-items:center;gap:12px;margin:4px 8px}
button{border:0;padding:2px 0;background:transparent;color:#555;font:inherit;font-size:11px;cursor:pointer}
button:disabled{opacity:.4;cursor:default}button:hover{color:black;text-decoration:underline}
output{color:#666}pre{margin:4px 8px;white-space:pre-wrap}
</style>
<pre id="numbers" hidden></pre><canvas aria-label="Animated BPL dots"></canvas>
<nav><button id="play" disabled>Pause</button><button id="step" disabled>Step</button>
<button id="restart" disabled>Restart</button><output>Loading BPL…</output></nav>
<script type="module">
const config = {"code": "4 4⍴0.5+0.5×1○(((⍳16)÷3)+(t÷8))", "scale": 1.0, "fps": 20.0, "numbers": false};
const canvas = document.querySelector('canvas'), ctx = canvas.getContext('2d');
const status = document.querySelector('output'), numbers = document.querySelector('#numbers');
const play = document.querySelector('#play'), step = document.querySelector('#step');
const restart = document.querySelector('#restart');
let worker, workerURL, pending, timer, busy=false, running=true, tick=0, scale=null, latest, restartRequested=false;
const workerSource = `
import init, {Session, configure} from 'https://cdn.jsdelivr.net/npm/basedpl@0.1.82/basedpl_wasm.js';
await init();
configure('https://cdn.jsdelivr.net/npm/basedpl@0.1.82/', () => {});
let session = new Session();
onmessage = ({data}) => {
 try {
 if (data.reset) { session.free(); session = new Session(); }
 session.bind({t: data.t});
 const value = session.eval(data.code);
 postMessage({value});
 } catch (e) { postMessage({error:e.display ?? e.message ?? String(e)}); }
};
postMessage({ready:true});`;
function fit() {
 if (window.frameElement) window.frameElement.style.height = (Math.ceil(document.body.getBoundingClientRect().height)+4)+'px';
}
function fail(error) {
 running=false; clearTimeout(timer); worker?.terminate();
 if(workerURL) URL.revokeObjectURL(workerURL);
 play.disabled=step.disabled=true; restart.disabled=false;
 status.textContent=String(error); fit();
}
function numeric(value) {
 if(typeof value==='number' || typeof value==='boolean') return {shape:[],data:[Number(value)]};
 if(!value || !Array.isArray(value.shape) || !value.data) throw Error('Use a rectangular real numeric array as the last expression.');
 const data=Array.from(value.data);
 if(data.some(x=>typeof x!=='number' && typeof x!=='boolean')) throw Error('Animated dots support real numbers and Booleans.');
 return {shape:value.shape,data:data.map(Number)};
}
function draw(value) {
 const a=numeric(value), {shape,data}=a, rank=shape.length;
 if(data.length>4096) throw Error('Animated dots support up to 4096 values.');
 const rows=rank<2?1:shape[rank-2], cols=rank===0?1:shape[rank-1];
 const panels=rank<3?1:shape.slice(0,-2).reduce((a,b)=>a*b,1);
 if(panels>64) throw Error('Animated dots support up to 64 grids.');
 const width=Math.max(40,cols*24+16), ph=rows*24+16, height=Math.max(40,panels*ph+(panels-1)*12);
 const dpr=devicePixelRatio||1;
 canvas.width=width*dpr; canvas.height=height*dpr; canvas.style.width=width+'px';
 canvas.style.height=height+'px'; ctx.scale(dpr,dpr);
 if(scale===null) scale=config.scale==='auto' ? Math.max(...data.filter(Number.isFinite).map(Math.abs),0)||1 : config.scale;
 ctx.fillStyle=ctx.strokeStyle='black';ctx.lineWidth=1;
 data.forEach((v,i)=>{
 const panel=Math.floor(i/(rows*cols)), local=i%(rows*cols);
 const x=20+(local%cols)*24, y=20+Math.floor(local/cols)*24+panel*(ph+12);
 if(!Number.isFinite(v)){ctx.fillText('×',x-3,y+4);return;}
 const r=v===0?1:8*Math.sqrt(Math.min(Math.abs(v)/scale,1));
 ctx.beginPath();ctx.arc(x,y,r,0,Math.PI*2);v<0?ctx.stroke():ctx.fill();
 });
 if(!data.length)ctx.fillText('∅',12,24);
 canvas.setAttribute('aria-label',`BPL dots, shape ${shape.join('

A sine wave changes the sizes gradually. `--scale 1` keeps a value of 1 at full size throughout. Without a scale, the first frame sets the scale for the animation, so changing magnitudes remain visible.

Each animated cell has its own BPL browser workspace. Put any supporting definitions in the same cell and finish with the numeric array you want to draw; the whole cell is evaluated for each frame. Use `t` as the supplied counter rather than assigning it yourself. Text containing "t" and names such as `total` do not trigger animation. Without the variable `t`, `--adots` gives ordinary static dots. `--numbers` also shows the current values.

The setup and examples are all in this dialogue. Animation loads the pinned BPL 0.1.82 browser interpreter from jsDelivr, so it needs internet access. This follows the browser animation approach used in First Look and the [official BPL animation docs](https://answerdotai.github.io/basedpl/web.html#how-the-gallery-works). This MVP supports rectangular arrays of real numbers and Booleans, up to 4096 values and 64 grids.